# Diagnostic Reasoning Assistant (DDXPlus) — Notebook 03: RAG & Explainability

**Grounded explanations for Top‑k diagnoses + next-question selection (TF‑IDF baseline), integrated with the saved token LR + IG policy**

**Author:** Julia Parnis   
**Date:** March 1, 2026  

## Project Overview

**Project goal:** Build an AI-powered diagnostic assistant that provides a *ranked differential* through iterative questioning, with a focus on improving detection of rare / overlooked conditions.

**Scope of this notebook:** Integrate a Retrieval-Augmented Generation (RAG) layer on top of the trained token-level Logistic Regression model to enable:
- Natural-language question phrasing (ask → patient answers → re-rank)
- Evidence-backed explainability (why this disease? what does the evidence mean?)
- Cited differential diagnosis summaries grounded in the DDXPlus knowledge base

This notebook focuses on *retrieval pipeline design, prompt engineering, and end-to-end session loop validation*.

**Planned (later phases):**
- Streamlit / API deployment (interactive web interface)
- Advanced question selection policies (reinforcement learning / active learning)
- Evaluation of RAG output quality (faithfulness, citation accuracy, clinical coherence)

**Inputs from Notebook 02:**
- `logreg_token_calibrated.joblib` — calibrated token model for probability ranking
- `preprocessors_token.joblib` — mlb_token, ohe, scaler, label_encoder
- `policy_artifacts.joblib` — p_e_given_d, evidence_bases, evidence_index
- `.py` modules in `assistant/` — types, encoding, inference, policy, artifacts

**Dataset:** DDXPlus (synthetic) — ~1.3M patient cases, 49 pathologies, evidence-based features  
**Source:** Hugging Face: `aai530-group6/ddxplus`  
**Primary reference:** Fansi Tchango et al. (2022)

> **Important:** DDXPlus is synthetic (generated from medical knowledge bases + an AD system).  
> This notebook is for research/education only; it does **not** produce a deployable clinical tool.

## Notebook 03 objective: RAG-powered explainability and iterative questioning

**Objective:** Build and validate a Retrieval-Augmented Generation layer on top of the trained token-level ML model to enable a natural-language diagnostic assistant, emphasising:
- Grounded question phrasing (questions generated from retrieved evidence descriptions, not hardcoded)
- Evidence-backed explainability (cited reasoning for the ranked differential)
- End-to-end session loop (ask → patient answers → re-rank → next question)
- Faithful retrieval (retrieved context is relevant, non-hallucinated, and traceable to source)

**Success criteria (for this notebook):**
- RAG pipeline runs end-to-end from a cold `DiagnosisSession` to a cited differential summary
- Questions are phrased in natural language and grounded in `release_evidences.json`
- Top-k differential is explained with at least one retrieved supporting fact per disease
- Session loop completes at least 5 question–answer turns without errors
- All prompts, retrievals, and outputs are logged and saved to `outputs/rag/`


# Notebook 03 — RAG Explanations for the DDXPlus Diagnostic Reasoning Assistant

## What we are building
We are adding a **local Retrieval-Augmented Generation (RAG)** layer that produces:
1) grounded explanations for **why the current Top‑k diagnoses** are ranked highly, and  
2) grounded explanations for **why the next question** was selected by the information‑gain policy.

## What stays fixed (no retraining)
- The ranking engine remains the **Logistic Regression token/value-level model** (972 token vocab; 978 total features with demographics + numeric complexity).
- The question policy remains **base-level (223 evidences)** using the likelihood table `p_e_given_d = P(e=1 | disease)`.
- For policy decisions, we prefer **calibrated probabilities** when available because information gain depends on meaningful uncertainty estimates.

## Data sources for retrieval (local, reproducible)
We build a retrieval corpus from:
- `release_evidences.json` (evidence IDs → question text, data type, default values, value meanings)
- `release_conditions.json` (condition names + symptom/antecedent evidence sets, severity, ICD hints)

Optional: we may add a small curated “mini-manual” layer (short markdown notes per condition) to improve explanation quality without requiring any external web access.

## Safety / scope
This project is **educational only** and uses the **synthetic DDXPlus dataset**. Explanations are phrased as learning support (how evidence patterns relate to diagnoses in this dataset and in general) and are not medical advice.

# Workplan (Notebook 03)

## 0) Setup and reproducibility
- Define paths (`data/`, `models/`, `outputs/rag/`) and create output folders.
- Load saved artifacts (preprocessors, calibrated token LR when available, policy table).
- Confirm feature shapes and that the pipeline runs without retraining.

## 1) Load dataset metadata (the retrieval “ground truth”)
- Load `release_evidences.json` (223 evidence bases).
- Load `release_conditions.json` (49 conditions).
- Build helper functions to format evidence definitions and condition summaries.

## 2) Build the local retrieval corpus (TF‑IDF baseline)
- Create documents for:
  - each evidence base (question text + type + default/value meanings)
  - each condition (name + key symptoms/antecedents listed in the metadata)
- Store document metadata (doc_type, evidence_id / condition_id, fields used).
- Fit a TF‑IDF vectorizer and persist the corpus + vectors to `outputs/rag/`.

## 3) (Optional) Add a curated mini-manual layer (49 conditions)
- Create one short markdown note per condition:
  - “what it is” (1–2 sentences)
  - “typical evidence themes” (based on dataset metadata)
  - “common confusions” (dataset-adjacent, educational framing)
- Include these notes as additional retrievable documents.

## 4) Retrieval API (clean interface used by the loop)
- Implement `retrieve(query, top_n)` returning:
  - ranked docs + similarity scores
  - doc metadata for grounding/provenance
- Add lightweight tests: deterministic results given fixed corpus.

## 5) Grounded explanation templates
- Template A: **“Why these Top‑k diagnoses now?”**
  - Use: posterior Top‑k + retrieved condition summaries + retrieved evidence definitions.
- Template B: **“Why ask this next question?”**
  - Use: chosen evidence base + question text + IG score + how it separates top diagnoses.

## 6) Integrate with the existing assistant loop
- Session state → token inference → posterior → IG policy → chosen question.
- Run retrieval to ground:
  - Top‑k explanation
  - next-question explanation
- Keep outputs consistent with the smoke test expectations (no retraining, same artifacts).

## 7) Demo trace + saved outputs
- Run a short simulated session (a few turns).
- Save:
  - retrieved contexts
  - explanations
  - a compact “trace log” for debugging / demo.

## 0) Setup & reproducibility

This section configures paths, seeds, display settings, and figure defaults (Zoom-friendly) consistent with previous notebooks.  
It also creates the new stage folders: `outputs/rag/` and `figures/rag/`.

In [1]:
# ============================================================
# 0.1 Imports + project-root check + src import
# ============================================================

from __future__ import annotations

import os
import sys
import random
import json
from pathlib import Path
from dataclasses import dataclass, field
import warnings
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# Visualization (match Notebook 02)
import matplotlib.pyplot as plt
from cycler import cycler

# Artifacts / ML utilities (needed later in this notebook)
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

# Silence noisy warnings (keep notebook readable)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
# ============================================================
# 0.1.1 Project root detection + path setup
# ============================================================

def find_project_root(start: Path | None = None) -> Path:
    """Find repo root by walking upward until data/ and models/ are found."""
    start = Path.cwd() if start is None else start.resolve()
    for p in [start] + list(start.parents):
        if (p / "data").exists() and (p / "models").exists():
            return p
    # Fallback: if running in a minimal environment, treat cwd as root
    return start


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"

print(f"✓ PROJECT_ROOT = {PROJECT_ROOT}")

# Ensure consistent relative paths throughout the notebook
os.chdir(PROJECT_ROOT)
print(f"✓ Working directory set to: {Path.cwd()}")

# Ensure src/ is importable when running from notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


✓ PROJECT_ROOT = c:\Julia\Ironhack\Week20-24_Final_project\diagnostic-reasoning-assistant
✓ Working directory set to: c:\Julia\Ironhack\Week20-24_Final_project\diagnostic-reasoning-assistant


In [3]:
# ============================================================
# 0.1.2 Quick check-up
# ============================================================

# Quick check-up
from src.artifacts import load_json  # noqa: E402
print("✓ src imports OK")

✓ src imports OK


In [4]:
# ============================================================
# 0.2 Reproducibility: seeds + version snapshot
# ============================================================

random_seed = 42

os.environ["PYTHONHASHSEED"] = str(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

print(f"✓ Random seed set to: {random_seed}")

# Capture a small environment snapshot (helps later when debugging)
import sklearn  # local import to keep import cell tidy

env_info = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "joblib": joblib.__version__,
}

print("Environment snapshot:")
for k, v in env_info.items():
    print(f"  - {k}: {v}")

✓ Random seed set to: 42
Environment snapshot:
  - python: 3.12.7
  - numpy: 2.4.2
  - pandas: 3.0.0
  - scikit_learn: 1.8.0
  - joblib: 1.5.3


In [5]:
# ============================================================
# 0.3 Config (Notebook-02 style) + create rag folders
# ============================================================

@dataclass
class Config:
    """
    Central configuration for this project.

    Folder strategy:
    - Fixed root folders: outputs/, models/, figures/
    - Subfolders: outputs/{stage}/, figures/{stage}/
    - File naming: stable filenames (no date tagging)
    """

    # ── Data paths ───────────────────────────────────────────
    data_dir: Path            = Path("data")
    evidence_map_path: Path   = Path("data/release_evidences.json")
    conditions_map_path: Path = Path("data/release_conditions.json")

    # ── Output folders ───────────────────────────────────────
    output_dir: Path  = Path("outputs")
    models_dir: Path  = Path("models")
    figures_dir: Path = Path("figures")

    # ── Reproducibility ──────────────────────────────────────
    random_seed: int = random_seed


config = Config()

# Ensure stage folders exist (Notebook 02 created eda/ml; Notebook 03 adds rag)
for stage in ["eda", "ml", "rag"]:
    (config.output_dir / stage).mkdir(parents=True, exist_ok=True)
    (config.figures_dir / stage).mkdir(parents=True, exist_ok=True)

config.models_dir.mkdir(parents=True, exist_ok=True)

print("✓ Directories ready:")
print(f"  - {config.output_dir / 'rag'}")
print(f"  - {config.figures_dir / 'rag'}")
print(f"  - {config.models_dir}")

✓ Directories ready:
  - outputs\rag
  - figures\rag
  - models


In [6]:
# ============================================================
# 0.4 Visualization Settings (same as Notebook 02)
# ============================================================

# Apply clean seaborn style (no seaborn import needed)
plt.style.use("seaborn-v0_8-white")

# IBM Design colorblind-safe palette
IBM_COLORS = {
    "blue": "#648FFF",
    "purple": "#785EF0",
    "magenta": "#DC267F",
    "orange": "#FE6100",
    "yellow": "#FFB000",
    "teal": "#06A39B",
    "gray": "#5F6368",
}

# Figure defaults (presentation-optimized)
plt.rcParams.update({
    # Figure size and resolution
    "figure.figsize": (6, 4),          # Good for 2+ figs per slide
    "figure.dpi": 120,                 # Screen display
    "savefig.dpi": 300,                # High-quality save

    # Font sizes (readable on Zoom)
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,

    # Appearance (subtle, professional)
    "axes.edgecolor": IBM_COLORS["gray"],
    "axes.linewidth": 1.2,
    "grid.color": "#D9D9D9",
    "grid.linewidth": 0.8,
    "grid.alpha": 0.6,
})

# Set IBM color cycle (for multi-line plots)
plt.rcParams["axes.prop_cycle"] = cycler(color=[
    IBM_COLORS["blue"],
    IBM_COLORS["teal"],
    IBM_COLORS["purple"],
    IBM_COLORS["magenta"],
    IBM_COLORS["orange"],
])

print("✓ Plot style configured (Notebook-02 consistent)")

# -------------------------
# Display settings (Notebook 02 style)
# -------------------------
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 4)

print("✓ Display settings configured")

✓ Plot style configured (Notebook-02 consistent)
✓ Display settings configured


In [7]:
# ============================================================
# 0.5 Save helpers 
# ============================================================

def save_fig(fig, filename: str, stage: str = "ml"):
    """
    Save figure into figures/{stage}/filename.

    stage: "eda" or "ml" or "rag"
    """
    out_dir = config.figures_dir / stage
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print(f"✓ Saved: {path}")


def save_table_csv(df: pd.DataFrame, filename: str, stage: str = "ml", index: bool = False):
    out_dir = config.output_dir / stage
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / filename
    df.to_csv(path, index=index)
    print(f"✓ Saved: {path}")

## Step 1 — Local retrieval corpus (DDXPlus metadata)

We build a **local** corpus from:
- `release_evidences.json` (223 evidence definitions: question text, data type, default value)
- `release_conditions.json` (49 condition entries: ICD10, severity, symptom/antecedent evidence sets)

This baseline corpus enables TF‑IDF retrieval (fast, reproducible) and is the foundation for grounded explanations.

In [8]:
# ============================================================
# 1.1 Load sources (DDXPlus mapping files)
# ============================================================

evidences_map = load_json(config.evidence_map_path)
conditions_map = load_json(config.conditions_map_path)

print(f"✓ Loaded release_evidences.json: {len(evidences_map)} evidences")
print(f"✓ Loaded release_conditions.json: {len(conditions_map)} conditions")

# Basic sanity checks (expected: 223 evidences, 49 conditions)
assert len(evidences_map) == 223, "Unexpected number of evidences (expected 223)."
assert len(conditions_map) == 49, "Unexpected number of conditions (expected 49)."

✓ Loaded release_evidences.json: 223 evidences
✓ Loaded release_conditions.json: 49 conditions


In [9]:
# ============================================================
# 1.1.1 Inspect mapping files (QC check)
# ============================================================

# Looking into evidences_map and conditions_map files

# ---- Example: one condition (lists evidence BASE codes) ----
example_condition = "Acute rhinosinusitis"
cond = conditions_map[example_condition]

print("\nCondition example:", example_condition)
print("  keys:", list(cond.keys()))
print("  icd10-id:", cond.get("icd10-id"))
print("  severity:", cond.get("severity"))
print("  #symptoms:", len(cond.get("symptoms", {})))
print("  #antecedents:", len(cond.get("antecedents", {})))
print("  symptom bases (first 12):", list(cond["symptoms"].keys())[:12])

# ---- Example: one evidence (contains question text + type + default) ----
example_evidence = "E_139"
ev = evidences_map[example_evidence]

print("\nEvidence example:", example_evidence)
print("  keys:", list(ev.keys()))
print("  question_en:", ev.get("question_en"))
print("  data_type:", ev.get("data_type"))
print("  default_value:", ev.get("default_value"))
print("  is_antecedent:", ev.get("is_antecedent"))


Condition example: Acute rhinosinusitis
  keys: ['condition_name', 'cond-name-fr', 'cond-name-eng', 'icd10-id', 'symptoms', 'antecedents', 'severity']
  icd10-id: j01
  severity: 4
  #symptoms: 12
  #antecedents: 10
  symptom bases (first 12): ['E_55', 'E_53', 'E_57', 'E_54', 'E_59', 'E_56', 'E_58', 'E_182', 'E_201', 'E_103', 'E_91', 'E_181']

Evidence example: E_139
  keys: ['name', 'code_question', 'question_fr', 'question_en', 'is_antecedent', 'default_value', 'value_meaning', 'possible-values', 'data_type']
  question_en: Do you have a known heart defect?
  data_type: B
  default_value: 0
  is_antecedent: True


In [10]:
# ============================================================
# 1.2 Build RAG corpus (documents + metadata)
# ============================================================

def _safe_str(x: Any) -> str:
    return "" if x is None else str(x)


def build_evidence_doc(base: str, meta: Dict[str, Any], *, max_value_examples: int = 12) -> Tuple[str, Dict[str, Any]]:
    """
    Build a compact text document for one evidence base.

    We intentionally do NOT dump huge value lists; we include only a small sample of value meanings.
    The full mapping stays available in `evidences_map` for UI rendering later.
    """
    q_en = _safe_str(meta.get("question_en")).strip()
    data_type = _safe_str(meta.get("data_type")).strip()  # B / C / M / ...
    default_value = meta.get("default_value")
    is_antecedent = bool(meta.get("is_antecedent", False))

    value_meaning = meta.get("value_meaning") or {}
    # Pull a small sample of English meanings (skip NA-like where possible)
    examples = []
    for v_code, v_meta in value_meaning.items():
        en = _safe_str((v_meta or {}).get("en")).strip()
        if not en or en.lower() == "na":
            continue
        examples.append(f"{v_code}={en}")
        if len(examples) >= max_value_examples:
            break

    lines = [
        f"Evidence base: {base}",
        f"Question (EN): {q_en}",
        f"Type: {data_type} | Antecedent: {is_antecedent} | Default value: {default_value}",
    ]
    if examples:
        lines.append("Example values: " + "; ".join(examples))

    doc_text = "\n".join([ln for ln in lines if ln.strip()])

    doc_meta = {
        "doc_type": "evidence",
        "base": base,
        "data_type": data_type,
        "is_antecedent": is_antecedent,
    }
    return doc_text, doc_meta


def build_condition_doc(
    condition_name: str,
    meta: Dict[str, Any],
    evidences_map: Dict[str, Any],
    *,
    max_evidence_list: int = 20,
) -> Tuple[str, Dict[str, Any]]:
    """
    Build a retrieval doc for one condition using:
    - ICD10 id
    - severity
    - linked symptom/antecedent evidence bases (with question text)
    """
    icd10 = _safe_str(meta.get("icd10-id")).strip()
    severity = meta.get("severity", None)

    symptoms = list((meta.get("symptoms") or {}).keys())
    antecedents = list((meta.get("antecedents") or {}).keys())

    def q(base: str) -> str:
        return _safe_str((evidences_map.get(base) or {}).get("question_en")).strip()

    symptom_lines = [f"- {b}: {q(b)}" for b in symptoms[:max_evidence_list] if q(b)]
    antecedent_lines = [f"- {b}: {q(b)}" for b in antecedents[:max_evidence_list] if q(b)]

    lines = [
        f"Condition: {condition_name}",
        f"ICD10: {icd10} | Severity: {severity}",
    ]
    if symptom_lines:
        lines.append("Common symptoms in dataset mapping:\n" + "\n".join(symptom_lines))
    if antecedent_lines:
        lines.append("Common antecedents in dataset mapping:\n" + "\n".join(antecedent_lines))

    doc_text = "\n".join(lines)

    doc_meta = {
        "doc_type": "condition",
        "condition_name": condition_name,
        "icd10": icd10,
        "severity": severity,
    }
    return doc_text, doc_meta


# ---- Build corpus ----
rag_docs: List[str] = []
rag_meta: List[Dict[str, Any]] = []

# 1) Evidence docs (223)
for base, meta in evidences_map.items():
    doc_text, doc_meta = build_evidence_doc(base, meta)
    rag_docs.append(doc_text)
    rag_meta.append(doc_meta)

# 2) Condition docs (49) — keys are already condition names
for condition_name, meta in conditions_map.items():
    doc_text, doc_meta = build_condition_doc(condition_name, meta, evidences_map)
    rag_docs.append(doc_text)
    rag_meta.append(doc_meta)

print(f"✓ RAG corpus built: {len(rag_docs)} documents")

# ---- Save a readable copy (JSONL) for inspection ----
corpus_path = config.output_dir / "rag" / "rag_corpus.jsonl"
with corpus_path.open("w", encoding="utf-8") as f:
    for text, meta in zip(rag_docs, rag_meta):
        rec = {"text": text, "meta": meta}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"✓ Saved corpus: {corpus_path}")

✓ RAG corpus built: 272 documents
✓ Saved corpus: outputs\rag\rag_corpus.jsonl


## Step 2 — Curated mini‑manual docs (49 conditions)

### Why this helps
Our current condition docs are already grounded, but they are mostly “lists.”
A mini‑manual gives the retriever richer, more structured text, which usually produces more professional explanations in:
- “Why these diagnoses?” (tie the posterior to plausible symptoms/antecedents)
- “Why this question next?” (tie policy selection to what distinguishes top conditions)

### What we will generate (grounded, local-only)
For each condition in `release_conditions.json`, we will write one markdown file containing:
- Condition name (+ ICD‑10 when present)
- *(If available)* a short, cited clinical enrichment summary from `src/condition_enrichments.py`  
  (educational only; used to improve wording and distinguish common confusions)
- DDXPlus severity value (dataset metadata)
- A “Symptoms” section listing the linked evidence questions (DDXPlus mapping)
- An “Antecedents / risk-factor questions” section listing linked evidence questions (DDXPlus mapping)
- A short educational disclaimer

These files will be saved to: `outputs/rag/mini_manual/` and will be added to the TF‑IDF corpus.


In [11]:
# ============================================================
# 2.0 Load clinical condition enrichments (optional)
#     - Used to enrich the mini-manual layer with short, cited summaries.
# ============================================================

# The project may store this file at src/condition_enrichments.py.
# We keep a fallback import to allow running in minimal environments.

CONDITION_ENRICHMENTS: Dict[str, str] = {}

try:
    from src.condition_enrichments import CONDITION_ENRICHMENTS as _CE  # type: ignore
    CONDITION_ENRICHMENTS = dict(_CE)
except Exception:
    try:
        from condition_enrichments import CONDITION_ENRICHMENTS as _CE  # type: ignore
        CONDITION_ENRICHMENTS = dict(_CE)
    except Exception:
        CONDITION_ENRICHMENTS = {}

if isinstance(CONDITION_ENRICHMENTS, dict) and len(CONDITION_ENRICHMENTS) > 0:
    missing = sorted(set(conditions_map.keys()) - set(CONDITION_ENRICHMENTS.keys()))
    extra = sorted(set(CONDITION_ENRICHMENTS.keys()) - set(conditions_map.keys()))
    if missing or extra:
        print("⚠ CONDITION_ENRICHMENTS keys do not match conditions_map.")
        print(f"  missing: {len(missing)}")
        print(f"  extra:   {len(extra)}")
    else:
        print(f"✓ Loaded CONDITION_ENRICHMENTS: {len(CONDITION_ENRICHMENTS)} conditions (keys match release_conditions.json)")
else:
    print("⚠ CONDITION_ENRICHMENTS not found. Mini-manual will use dataset metadata only.")


✓ Loaded CONDITION_ENRICHMENTS: 49 conditions (keys match release_conditions.json)


In [12]:
# ============================================================
# 2.1 Mini-manual doc builder (DDXPlus metadata → Markdown + clinical enrichments)
# ============================================================

import re
from typing import Any, Dict, List, Tuple


def slugify(text: str, max_len: int = 80) -> str:
    """Make a filesystem-friendly slug from a condition name."""
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "-", text)
    text = re.sub(r"-{2,}", "-", text).strip("-")
    return text[:max_len] if len(text) > max_len else text


def sort_evidence_codes(codes: List[str]) -> List[str]:
    """Sort evidence codes like E_2, E_10, E_100 numerically when possible."""

    def _key(code: str) -> int:
        m = re.search(r"E_(\d+)", code)
        return int(m.group(1)) if m else 10**9

    return sorted(codes, key=_key)


def evidence_bullet(base: str, evidences_map: Dict[str, Any]) -> str:
    """Build one bullet line for an evidence base using release_evidences.json."""
    meta = evidences_map.get(base, {})
    q_en = meta.get("question_en") or base
    data_type = meta.get("data_type")
    if data_type:
        return f"- **{base}** ({data_type}): {q_en}"
    return f"- **{base}**: {q_en}"


def bullets_for_codes(
    codes: List[str],
    evidences_map: Dict[str, Any],
    *,
    max_items: int = 15,
) -> str:
    """Render a (possibly truncated) bullet list for evidence base codes."""
    codes_sorted = sort_evidence_codes(list(codes))
    shown = codes_sorted[:max_items]
    lines = [evidence_bullet(c, evidences_map) for c in shown]

    if len(codes_sorted) > max_items:
        lines.append(f"- … ({len(codes_sorted) - max_items} more omitted for brevity)")
    return "\n".join(lines)


def _clean_condition_enrichment(condition_name: str) -> str:
    """Return the enrichment markdown block for this condition (best-effort)."""
    enrichments = globals().get("CONDITION_ENRICHMENTS", {})
    if not isinstance(enrichments, dict):
        return ""
    raw = str(enrichments.get(condition_name, "") or "").strip()
    if not raw:
        return ""

    # Many entries start with "**CONDITION: ...**" which is redundant with our '# Condition' header.
    lines = [ln.rstrip() for ln in raw.splitlines()]
    if lines and lines[0].strip().startswith("**CONDITION:"):
        lines = lines[1:]
    return "\n".join(lines).strip()


def build_condition_manual_doc(
    condition_name: str,
    meta: Dict[str, Any],
    evidences_map: Dict[str, Any],
    *,
    max_symptoms: int = 15,
    max_antecedents: int = 15,
) -> Tuple[str, Dict[str, Any]]:
    """Create a markdown mini-manual doc for one condition (educational; dataset-grounded + cited enrichments)."""
    icd10 = meta.get("icd10-id", "")
    severity = meta.get("severity", None)

    symptom_codes = list((meta.get("symptoms") or {}).keys())
    antecedent_codes = list((meta.get("antecedents") or {}).keys())

    enrichment_md = _clean_condition_enrichment(condition_name)

    md: List[str] = []
    md.append(f"# {condition_name}")
    md.append("")

    # Prefer to place clinical context near the top so the assistant can surface a short blurb.
    if enrichment_md:
        md.append(enrichment_md)
        md.append("")

    md.append(
        "**Educational note:** This mini-manual combines (a) curated clinical summaries (with cited sources) and "
        "(b) DDXPlus synthetic metadata mappings. It is not medical advice and is used only to explain model behavior."
    )
    md.append("")

    md.append("## Dataset metadata")
    if icd10:
        md.append(f"- ICD‑10: `{icd10}`")
    if severity is not None:
        md.append(f"- DDXPlus severity (dataset field): `{severity}`")
    md.append("")

    md.append("## Symptoms linked to this condition (DDXPlus mapping)")
    md.append(bullets_for_codes(symptom_codes, evidences_map, max_items=max_symptoms) or "- (none listed)")
    md.append("")

    md.append("## Antecedents / risk-factor questions linked to this condition (DDXPlus mapping)")
    md.append(bullets_for_codes(antecedent_codes, evidences_map, max_items=max_antecedents) or "- (none listed)")
    md.append("")

    md.append("## Notes for the assistant")
    md.append("- These are **base evidence codes** (question concepts). The predictor still uses token/value encoding.")
    md.append("- The question policy selects among base codes using an expected usefulness score; RAG only provides wording/explanations.")

    doc_text = "\n".join(md)

    doc_meta = {
        "doc_type": "manual",
        "condition_name": condition_name,
        "icd10": icd10,
        "severity": severity,
        "n_symptoms": len(symptom_codes),
        "n_antecedents": len(antecedent_codes),
        "has_clinical_enrichment": bool(enrichment_md),
    }
    return doc_text, doc_meta


In [13]:
# ============================================================
# 2.2 Write mini-manual docs to disk + load back into memory
# ============================================================

import json
import pandas as pd

# --- Preconditions: these should already exist from Steps 0–1 ---
assert "config" in globals(), "Expected 'config' from setup cells."
assert "conditions_map" in globals(), "Expected 'conditions_map' loaded from release_conditions.json."
assert "evidences_map" in globals(), "Expected 'evidences_map' loaded from release_evidences.json."

# --- Output folder following the project folder strategy ---
manual_dir = config.output_dir / "rag" / "mini_manual"
manual_dir.mkdir(parents=True, exist_ok=True)

rows = []
index = {}

# Stable ordering for reproducibility
for condition_name in sorted(conditions_map.keys()):
    meta = conditions_map[condition_name]
    doc_text, doc_meta = build_condition_manual_doc(
        condition_name,
        meta,
        evidences_map,
        max_symptoms=18,
        max_antecedents=18,
    )

    fname = f"{slugify(condition_name)}.md"
    path = manual_dir / fname
    path.write_text(doc_text, encoding="utf-8")

    index[condition_name] = fname
    rows.append(
        {
            "condition_name": condition_name,
            "icd10": doc_meta.get("icd10"),
            "severity": doc_meta.get("severity"),
            "n_symptoms": doc_meta.get("n_symptoms"),
            "n_antecedents": doc_meta.get("n_antecedents"),
            "has_clinical_enrichment": bool(doc_meta.get("has_clinical_enrichment")),
            "manual_file": fname,
        }
    )

manual_index_path = manual_dir / "index.json"
manual_index_path.write_text(json.dumps(index, indent=2), encoding="utf-8")

manual_summary = pd.DataFrame(rows).sort_values(["severity", "condition_name"], ascending=[True, True])
save_table_csv(manual_summary, "mini_manual_summary.csv", stage="rag", index=False)

print(f"✓ Wrote {len(rows)} mini-manual files to: {manual_dir}")
print(f"✓ Wrote manual index to: {manual_index_path}")
print("✓ Saved summary CSV to outputs/rag/mini_manual_summary.csv")

# Clinical enrichment QC (best-effort)
n_enriched = int(manual_summary["has_clinical_enrichment"].sum())
print(f"✓ Clinical enrichment present in mini-manual docs: {n_enriched}/{len(rows)}")

if isinstance(globals().get("CONDITION_ENRICHMENTS", {}), dict) and len(globals().get("CONDITION_ENRICHMENTS", {})) > 0:
    # If the enrichments dict is present, we expect full coverage.
    assert n_enriched == len(rows), "Expected clinical enrichment for all conditions."

# --- Load mini-manual back into memory for later use in explanations ---
mini_manual: Dict[str, str] = {}
for cond_name, fname in index.items():
    p = manual_dir / fname
    if p.exists():
        mini_manual[cond_name] = p.read_text(encoding="utf-8")

print(f"✓ Loaded mini_manual into memory: {len(mini_manual)} condition docs")

manual_summary.head(8)


✓ Saved: outputs\rag\mini_manual_summary.csv
✓ Wrote 49 mini-manual files to: outputs\rag\mini_manual
✓ Wrote manual index to: outputs\rag\mini_manual\index.json
✓ Saved summary CSV to outputs/rag/mini_manual_summary.csv
✓ Clinical enrichment present in mini-manual docs: 49/49
✓ Loaded mini_manual into memory: 49 condition docs


,condition_name,icd10,severity,n_symptoms,n_antecedents,has_clinical_enrichment,manual_file
4,Acute pulmonary edema,J81.0,1,15,6,True,acute-pulmonary-edema.md
7,Anaphylaxis,T78.0,1,25,3,True,anaphylaxis.md
19,Ebola,a98.4,1,10,3,True,ebola.md
26,Larygospasm,J38.5,1,1,6,True,larygospasm.md
35,Possible NSTEMI / STEMI,I21,1,12,11,True,possible-nstemi-stemi.md
1,Acute dystonic reactions,G24.02,2,8,4,True,acute-dystonic-reactions.md
10,Boerhaave,K22.3,2,11,2,True,boerhaave.md
18,Croup,J05.0,2,7,3,True,croup.md


In [14]:
# ============================================================
# 2.3 Build TF‑IDF retriever (canonical) + save artifacts
#     Canonical path: outputs/rag/tfidf_retriever.joblib (321 docs)
# ============================================================

# --- Preconditions: doc builders should already exist from Step 1.2 ---
assert "build_condition_doc" in globals(), "Expected 'build_condition_doc' from earlier cells."
assert "build_evidence_doc" in globals(), "Expected 'build_evidence_doc' from earlier cells."
assert "config" in globals(), "Expected 'config' from setup cells."

# 1) Base corpus: evidences + conditions
docs: List[str] = []
meta: List[Dict[str, Any]] = []

for base in sorted(evidences_map.keys(), key=lambda x: int(x.split("_")[1])):
    text, m = build_evidence_doc(base, evidences_map[base])
    docs.append(text)
    meta.append(m)

for condition_name in sorted(conditions_map.keys()):
    text, m = build_condition_doc(condition_name, conditions_map[condition_name], evidences_map)
    docs.append(text)
    meta.append(m)

# 2) Add mini-manual docs (49) — prefer disk, fallback to in-memory dict
manual_dir = config.output_dir / "rag" / "mini_manual"
index_path = manual_dir / "index.json"

n_manual_added = 0
if index_path.exists():
    index = json.loads(index_path.read_text(encoding="utf-8"))
    for condition_name in sorted(index.keys()):
        f = index[condition_name]
        p = manual_dir / f
        if p.exists():
            docs.append(p.read_text(encoding="utf-8"))
            meta.append({"doc_type": "manual", "condition_name": condition_name, "path": str(p)})
            n_manual_added += 1
elif "mini_manual" in globals() and isinstance(mini_manual, dict) and len(mini_manual) > 0:
    for condition_name in sorted(mini_manual.keys()):
        docs.append(str(mini_manual[condition_name]))
        meta.append({"doc_type": "manual", "condition_name": condition_name, "path": None})
        n_manual_added += 1
else:
    print("⚠ mini-manual not found on disk and not in memory. Retriever will exclude manual docs.")

print(f"Corpus docs: {len(docs)} (evidence + condition + manual={n_manual_added})")

# Canonical corpus size check: 223 evidences + 49 conditions + 49 manuals = 321
if n_manual_added == 49:
    assert len(docs) == 321, "Expected 321 docs in canonical TF-IDF corpus."

# 3) Fit TF‑IDF retriever (baseline)
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=6000,
    ngram_range=(1, 2),
)
X_docs = vectorizer.fit_transform(docs)

# 4) Save artifact for later integration with the assistant loop
retriever_artifact = {
    "vectorizer": vectorizer,
    "X_docs": X_docs,
    "docs": docs,
    "meta": meta,
}

retriever_path = config.output_dir / "rag" / "tfidf_retriever.joblib"
joblib.dump(retriever_artifact, retriever_path)

print(f"✓ Saved TF‑IDF retriever artifact to: {retriever_path}")

# 5) Quick QC check query (should return some manual/condition/evidence docs)
test_query = "Acute rhinosinusitis"
qv = vectorizer.transform([test_query])
sims = cosine_similarity(qv, X_docs).ravel()
top_idx = sims.argsort()[::-1][:5]

print(f"\nTop results for query='{test_query}':")
for i in top_idx:
    print(f"  score={float(sims[i]):.4f}  meta={meta[i]}")


Corpus docs: 321 (evidence + condition + manual=49)
✓ Saved TF‑IDF retriever artifact to: outputs\rag\tfidf_retriever.joblib

Top results for query='Acute rhinosinusitis':
  score=0.1688  meta={'doc_type': 'manual', 'condition_name': 'Acute rhinosinusitis', 'path': 'outputs\\rag\\mini_manual\\acute-rhinosinusitis.md'}
  score=0.1499  meta={'doc_type': 'manual', 'condition_name': 'Chronic rhinosinusitis', 'path': 'outputs\\rag\\mini_manual\\chronic-rhinosinusitis.md'}
  score=0.1429  meta={'doc_type': 'condition', 'condition_name': 'Acute rhinosinusitis', 'icd10': 'j01', 'severity': 4}
  score=0.0594  meta={'doc_type': 'condition', 'condition_name': 'Chronic rhinosinusitis', 'icd10': 'j32', 'severity': 5}
  score=0.0507  meta={'doc_type': 'manual', 'condition_name': 'Acute COPD exacerbation / infection', 'path': 'outputs\\rag\\mini_manual\\acute-copd-exacerbation-infection.md'}


## Step 3 — RAG explanation building blocks (templates + grounded snippets)

We now connect the pieces into the interactive flow:

**session → token inference → posterior → IG policy → chosen question → RAG explanations**

We will implement two grounded templates:
1) **Why these Top‑k diagnoses now?**
2) **Why ask this next question?**

RAG stays strictly as a *wording/explanation* layer: it does not modify model probabilities.

In [15]:
# ============================================================
# 3.1 Load artifacts (no retraining) + import src utilities
# ============================================================

from pathlib import Path
import joblib
import numpy as np

from src.artifacts import load_json, load_preprocessors, resolve_best_token_model, load_policy_artifacts
from src.types import DiagnosisSession, FeatureSpec
from src.inference import predict_topk_from_session_token
from src.policy import choose_next_question_info_gain

# Paths (project-root relative)
MODELS_DIR = Path("models")
DATA_DIR = Path("data")
OUT_RAG = Path("outputs") / "rag"

# Mapping files (for question text and condition names)
evidences_map = load_json(DATA_DIR / "release_evidences.json")
conditions_map = load_json(DATA_DIR / "release_conditions.json")

# Preprocessors (token)
# Preferred artifact: models/preprocessors.joblib (bundle containing mlb_base + mlb_token + metadata)
# Legacy fallback:     models/preprocessors_token.joblib
preprocessors_candidates = [
    MODELS_DIR / "preprocessors.joblib",
    MODELS_DIR / "preprocessors_token.joblib",
]

preprocessors_path = None
for p in preprocessors_candidates:
    if p.exists():
        preprocessors_path = p
        break

if preprocessors_path is None:
    raise FileNotFoundError(
        "No preprocessor artifact found. Expected one of: "
        + ", ".join(str(p) for p in preprocessors_candidates)
    )

pre = load_preprocessors(preprocessors_path)

if "mlb_token" not in pre:
    raise KeyError(
        "Expected 'mlb_token' in preprocessors artifact. "
        "Use models/preprocessors.joblib (preferred) or models/preprocessors_token.joblib."
    )

mlb_token = pre["mlb_token"]
ohe = pre["ohe"]
scaler = pre["scaler"]
label_encoder = pre["label_encoder"]
CAT_COLS = list(pre["CAT_COLS"])
NUM_COLS = list(pre["NUM_COLS"])

# Token model (prefer calibrated when present)
model_token = resolve_best_token_model(MODELS_DIR)

# Policy artifacts
pol = load_policy_artifacts(MODELS_DIR / "policy_artifacts.joblib")
p_e_given_d = pol["p_e_given_d"]
evidence_bases = list(pol["evidence_bases"])

# FeatureSpec for inference (kept explicit to avoid mismatch)
feature_spec = FeatureSpec(
    evidence_mode="token",
    cat_cols=CAT_COLS,
    num_cols=NUM_COLS,
    age_group_func=None,
)

# Retriever artifact (built in previous steps)
retriever_path = OUT_RAG / "tfidf_retriever.joblib"
retr = joblib.load(retriever_path)

print("✓ Loaded artifacts:")
print("  - preprocessors:", preprocessors_path)
print("  - mlb_token vocab:", len(mlb_token.classes_))
print("  - model_token:", type(model_token).__name__)
print("  - policy p_e_given_d:", np.array(p_e_given_d).shape, "with", len(evidence_bases), "bases")
print("  - retriever:", retriever_path)


✓ Loaded artifacts:
  - preprocessors: models\preprocessors.joblib
  - mlb_token vocab: 972
  - model_token: CalibratedClassifierCV
  - policy p_e_given_d: (49, 223) with 223 bases
  - retriever: outputs\rag\tfidf_retriever.joblib


In [16]:
# ============================================================
# 3.2 Retrieval helper (TF‑IDF artifact)
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity


def retrieve_snippets(query: str, *, k: int = 4, doc_type: str | None = None):
    """Retrieve top-k snippets from the TF‑IDF corpus.

    doc_type can be:
      - None (no filter)
      - 'manual' | 'condition' | 'evidence' (matches meta['doc_type'])
    """
    vectorizer = retr["vectorizer"]
    X_docs = retr["X_docs"]
    docs = retr["docs"]
    meta = retr["meta"]

    qv = vectorizer.transform([query])
    sims = cosine_similarity(qv, X_docs).ravel()
    order = sims.argsort()[::-1]

    hits = []
    for idx in order:
        m = meta[idx]
        if doc_type is not None and m.get("doc_type") != doc_type:
            continue
        hits.append({"score": float(sims[idx]), "meta": m, "text": docs[idx]})
        if len(hits) >= k:
            break
    return hits


### 3.3 User-facing explanations (plain language, medically grounded)

This notebook uses a **two-part approach**:
1) a statistical ranking engine produces a short list of likely diagnoses based on the information recorded so far, and  
2) a retrieval layer provides **human-readable wording** and **grounded explanations** using local documents (DDXPlus mappings + optional mini‑manual notes).

**Audience & tone (important):**
- Explanations are written for medical students or non-technical learners.
- Technical ML terms are avoided (e.g., “token model”, “logistic regression”, “information gain”).
- Wording uses *we* voice or passive voice (avoid “you”), and keeps a calm, educational tone.

**Grounding rules (no hallucinations):**
- The assistant only explains using:
  - recorded findings already present in the session, and
  - retrieved context from the local corpus built from DDXPlus mapping files, plus
  - optional short “mini‑manual” summaries that we curate (1–2 sentences per condition) with references.
- If a clinical fact is not present in the curated mini‑manual (or the retrieved corpus), it is not stated as a fact.

**What the assistant explains (two templates):**

**A) “Why these Top‑k diagnoses now?”**
- A short list of the current leading diagnoses (Top‑k).
- For each diagnosis:
  - 1 short “what this condition is” line (from mini‑manual if available; otherwise a minimal label),
  - 2–3 “why it is on the list now” points tied to the recorded findings so far,
  - optionally: 1 sentence on what additional information would help separate it from nearby alternatives.

**B) “Why ask this next question?”**
- The next question is chosen because it is expected to **separate the leading possibilities quickly**.
- Explanation is phrased clinically:
  - “This question helps because the answer often differs between the leading conditions.”
  - “A ‘yes’ answer would support …; a ‘no’ answer would make … less likely.”
- If probabilities are shown, they are presented as *model-based estimates* (not certainty).

**Safety / scope disclaimer:**
- This is an educational tool. It does not replace clinical judgment or medical advice.

In [17]:
# ============================================================
# 3.4 Helpers: readable findings + overlap checks + safe token picking
# ============================================================

from typing import Any, Dict, List, Optional, Set

from src.types import DiagnosisSession, parse_token


# Cached token vocabulary (safe even if this cell is re-run)
_TOKEN_VOCAB: Set[str] = set(getattr(mlb_token, "classes_", []))


def _value_to_english(base: str, val: str, evidences_map: Dict[str, Any]) -> str:
    """Best-effort mapping from coded values (e.g., V_29) to English meaning."""
    vm = evidences_map.get(base, {}).get("value_meaning", {}) or {}
    if isinstance(vm, dict) and val in vm and isinstance(vm[val], dict):
        return (vm[val].get("en") or val)
    return val


def token_to_readable(token: str, evidences_map: Dict[str, Any]) -> str:
    """Convert an evidence token into a short, human-readable finding."""
    base, val = parse_token(token)
    q = (evidences_map.get(base, {}).get("question_en") or base).strip()

    if val is None:
        return f"{q} (reported)"
    return f"{q} (value: {_value_to_english(base, val, evidences_map)})"


def session_known_positives(
    session: DiagnosisSession,
    evidences_map: Dict[str, Any],
    *,
    max_items: int = 8,
) -> List[str]:
    items = sorted(session.pos_items)
    readable = [token_to_readable(t, evidences_map) for t in items[:max_items]]
    if len(items) > max_items:
        readable.append(f"(+ {len(items) - max_items} more positive findings)")
    return readable


def condition_overlap_bases(
    condition_name: str,
    session: DiagnosisSession,
    conditions_map: Dict[str, Any],
) -> Dict[str, List[str]]:
    """Return overlap of session positives with condition-linked symptom/antecedent bases."""
    cond = conditions_map.get(condition_name, {}) or {}
    symptom_bases = set((cond.get("symptoms", {}) or {}).keys())
    antecedent_bases = set((cond.get("antecedents", {}) or {}).keys())

    pos_bases = session.pos_bases
    return {
        "symptoms": sorted(pos_bases.intersection(symptom_bases)),
        "antecedents": sorted(pos_bases.intersection(antecedent_bases)),
    }


def valid_positive_token_for_base(
    base: str,
    evidences_map: Dict[str, Any],
    token_vocab: Set[str],
) -> Optional[str]:
    """Pick a positive/non-default token for a base that exists in the trained token vocab."""
    if base in token_vocab:
        return base  # binary evidence token

    meta = evidences_map.get(base, {})
    default_val = meta.get("default_value", None)
    poss = meta.get("possible-values") or meta.get("possible_values") or []

    # Try each possible value except the default (string and raw forms)
    for v in poss:
        if default_val is not None and v == default_val:
            continue
        tok = f"{base}={v}"
        if tok in token_vocab:
            return tok

    for v in poss:
        if default_val is not None and v == default_val:
            continue
        tok = f"{base}={str(v)}"
        if tok in token_vocab:
            return tok

    return None


In [18]:
# ============================================================
# 3.4.1 Build evidence_index (policy table lookup)
# ============================================================

# evidence_bases is the list of base evidence codes in the SAME order as columns in p_e_given_d
# p_e_given_d has shape (n_conditions, n_evidence_bases)
assert p_e_given_d.shape[1] == len(evidence_bases), "Policy table columns mismatch (p_e_given_d vs evidence_bases)"

# Map evidence base -> column index in p_e_given_d
evidence_index = {base: j for j, base in enumerate(evidence_bases)}

print("✓ Built evidence_index")
print("  n_bases:", len(evidence_index))
print("  sample:", list(evidence_index.items())[:5])

✓ Built evidence_index
  n_bases: 223
  sample: [('E_0', 0), ('E_1', 1), ('E_10', 2), ('E_100', 3), ('E_101', 4)]


In [19]:
# ============================================================
# 3.5 Canonical explanation functions (Top‑k + next question)
# ============================================================

from typing import Any, Dict, List, Optional, Tuple


def _snippet_first_lines(text: str, *, max_lines: int = 6) -> str:
    lines = [ln.strip() for ln in (text or "").splitlines() if ln.strip()]
    return " | ".join(lines[:max_lines])


def _mini_manual_blurb(md: str, *, max_lines: int = 3) -> str:
    """Extract a short blurb from a markdown mini-manual doc (best-effort)."""
    lines = [ln.strip() for ln in (md or "").splitlines() if ln.strip()]
    if not lines:
        return ""

    # Drop markdown headings (we want the clinical/context text, not section titles)
    lines = [ln for ln in lines if not ln.startswith("#")]
    if not lines:
        return ""

    # Prefer to start at an explicit overview if present (common in condition enrichments)
    for i, ln in enumerate(lines):
        if ln.startswith("**Overview:**") or ln.lower().startswith("overview:"):
            lines = lines[i:]
            break

    # Avoid leading boilerplate if present
    if lines and lines[0].lower().startswith("**educational note:**"):
        lines = lines[1:]

    return " ".join(lines[:max_lines])


def _get_retriever_fn(art: Dict[str, Any]):
    """Return a callable retrieve_snippets(...) function (ART-first, then notebook global)."""
    fn = None
    if isinstance(art, dict):
        fn = art.get("retrieve_snippets", None)
    if callable(fn):
        return fn
    fn = globals().get("retrieve_snippets", None)
    return fn if callable(fn) else None


def explain_topk_diagnoses(
    session: DiagnosisSession,
    topk: List[Tuple[str, float]],
    *,
    art: Dict[str, Any],
    max_known: int = 6,
) -> str:
    """Explain why the current Top‑k diagnoses are leading (dataset-grounded; educational)."""
    evidences_map = art["evidences_map"]
    conditions_map = art["conditions_map"]
    mini_manual = art.get("mini_manual") or {}
    retr_fn = _get_retriever_fn(art)

    known = session_known_positives(session, evidences_map, max_items=max_known)

    lines: List[str] = []
    lines.append("Educational explanation (not medical advice).")
    if known:
        lines.append("What we know so far (positive findings):")
        for s in known:
            lines.append(f"  - {s}")
    else:
        lines.append("What we know so far: no positive findings recorded yet.")

    lines.append("")
    lines.append("Why we currently rank these diagnoses near the top (dataset-grounded):")
    for rank, (name, p) in enumerate(topk, start=1):
        overlap = condition_overlap_bases(name, session, conditions_map)
        ov_sym = overlap["symptoms"][:4]
        ov_ant = overlap["antecedents"][:3]

        # Prefer a short mini-manual blurb if available (still dataset-grounded)
        manual_blurb = ""
        if isinstance(mini_manual, dict) and name in mini_manual:
            manual_blurb = _mini_manual_blurb(str(mini_manual.get(name, "")), max_lines=3)

        # Fallback: retrieve a short condition/manual snippet from the local corpus
        snippet_txt = ""
        if manual_blurb == "" and retr_fn is not None:
            hits = retr_fn(name, k=1, doc_type="manual") or retr_fn(name, k=1, doc_type="condition")
            if hits:
                snippet_txt = _snippet_first_lines(hits[0].get("text", ""), max_lines=6)

        lines.append(f"{rank}) {name} (estimated likelihood ≈ {p:.3f})")
        if manual_blurb:
            lines.append(f"   Mini-manual: {manual_blurb}")
        elif snippet_txt:
            lines.append(f"   Retrieved context (local corpus): {snippet_txt}")

        if ov_sym or ov_ant:
            parts = []
            if ov_sym:
                parts.append(f"overlaps with listed symptom concepts: {', '.join(ov_sym)}")
            if ov_ant:
                parts.append(f"overlaps with listed history/risk concepts: {', '.join(ov_ant)}")
            lines.append("   Dataset grounding: " + "; ".join(parts) + ".")
        else:
            lines.append(
                "   Dataset grounding: no direct overlap found with the recorded positives "
                "(this can happen early; additional questions may be needed)."
            )

    return "\n".join(lines)


def explain_next_question(
    session: DiagnosisSession,
    candidates: List[Tuple[str, float, float, str]],
    topk: List[Tuple[str, float]],
    *,
    art: Dict[str, Any],
    top_show: int = 3,
) -> str:
    """Explain why the next question was selected and how it separates top diagnoses."""
    if not candidates:
        return "No next-question candidates found (all evidence bases may already be asked)."

    evidences_map = art["evidences_map"]
    label_encoder = art["label_encoder"]
    p_e_given_d = art["p_e_given_d"]
    evidence_index = art["evidence_index"]
    retr_fn = _get_retriever_fn(art)

    base, score, p_yes, q_text = candidates[0]
    meta = evidences_map.get(base, {}) or {}
    q_final = (meta.get("question_en") or q_text or base).strip()

    lines: List[str] = []
    lines.append("Why we ask this next question (educational explanation; not medical advice):")
    lines.append(f"- Next question: {q_final}")
    lines.append(
        "- We selected this question because it is expected to be especially helpful for telling the leading options apart."
    )
    lines.append(f"- In this dataset, the chance of a 'yes' answer (given the current ranking) is ≈ {p_yes:.2f}.")

    # Show how it separates the top diagnoses using p(e=1|d) for top-k
    lines.append("")
    lines.append("How this question tends to differ across the leading diagnoses (dataset frequencies):")
    top_names = [n for n, _ in topk[:top_show]]
    j = evidence_index.get(base, None)
    if j is None:
        lines.append("- (This evidence base is not in the policy table index.)")
        return "\n".join(lines)

    for name in top_names:
        idx = int(label_encoder.transform([name])[0])
        pe = float(p_e_given_d[idx, j])
        lines.append(f"- {name}: ~{pe:.2f} of cases have this as 'yes'")

    # Retrieve evidence snippet (definition)
    if retr_fn is not None:
        hits = retr_fn(base, k=1, doc_type="evidence")
        if hits:
            snippet_txt = _snippet_first_lines(hits[0].get("text", ""), max_lines=5)
            if snippet_txt:
                lines.append("")
                lines.append(f"Evidence definition (retrieved from local corpus): {snippet_txt}")

    # Internal numeric score is kept but not called "information gain"
    lines.append("")
    lines.append(f"(Internal usefulness score ≈ {score:.4f}; higher generally means more expected usefulness.)")

    return "\n".join(lines)


## 4) End-to-end “assistant turn”: model → next question → grounded explanation

In this section, we connect the saved **token Logistic Regression** model and the **base-level question policy** to our **local retrieval corpus**.
The result is a reusable function that produces:
- Top‑k diagnoses (with probabilities)
- The next best question to ask
- Two user-facing explanations grounded in retrieved dataset context

**Important:** This is an educational prototype and does not provide medical advice.

In [20]:
# ============================================================
# 4.1 Validation check: artifacts already loaded (no re-loading)
# ============================================================

required_names = [
    # maps
    "evidences_map", "conditions_map",
    # preprocessors + feature spec
    "mlb_token", "ohe", "scaler", "label_encoder", "CAT_COLS", "NUM_COLS", "feature_spec",
    # model + policy
    "model_token", "p_e_given_d", "evidence_bases",
    # retriever
    "retr",
]

missing = [n for n in required_names if n not in globals()]
assert not missing, (
    "Missing required objects for the end-to-end loop:\n"
    f"{missing}\n\n"
    "Run the single 'Load artifacts (no retraining)' cell first (Step 3.1)."
)

# Quick shape checks (helpful for catching subtle mismatches)
import numpy as np

assert len(getattr(mlb_token, "classes_", [])) == 972, "mlb_token vocab expected to be 972."
assert np.asarray(p_e_given_d).shape[1] == len(evidence_bases), "p_e_given_d columns must match evidence_bases."

print("✓ QC checks passed — ready for the end-to-end assistant loop.")

✓ QC checks passed — ready for the end-to-end assistant loop.


In [21]:
# ============================================================
# 4.2 assistant_turn: session → posterior → policy → next question
#      (single canonical implementation; ART-driven)
# ============================================================

from typing import Any, Dict

import numpy as np

from src.inference import predict_topk_from_session_token
from src.policy import choose_next_question_info_gain, update_posterior_with_answer


def assistant_turn(
    session: DiagnosisSession,
    *,
    art: Dict[str, Any],
    k: int = 5,
    top_n_questions: int = 6,
    use_tempered_negatives_for_policy: bool = True,
    beta_neg: float = 0.7,
) -> Dict[str, Any]:
    """Run one assistant step (educational; no retraining).

    Steps:
      1) ML inference (token model) → posterior over conditions
      2) Optional conservative incorporation of explicitly negative answers (policy-only)
      3) Policy ranking of next question candidates (base evidences)
      4) Return a stable payload for demos / UI
    """

    # --- 1) Model posterior from token LR (calibrated preferred via resolve_best_token_model) ---
    topk_model, posterior_model = predict_topk_from_session_token(
        session,
        art["model_token"],
        mlb_token=art["mlb_token"],
        ohe=art["ohe"],
        scaler=art["scaler"],
        label_encoder=art["label_encoder"],
        feature_spec=art["feature_spec"],
        k=k,
    )
    posterior_for_policy = np.asarray(posterior_model, dtype=float).copy()

    # --- 2) Optional: incorporate negatives conservatively for policy only ---
    if use_tempered_negatives_for_policy and session.neg_bases:
        for base in sorted(session.neg_bases):
            posterior_for_policy = update_posterior_with_answer(
                posterior_for_policy,
                base=base,
                answer="no",
                p_e_given_d=art["p_e_given_d"],
                evidence_index=art["evidence_index"],
                beta=beta_neg,
            )

    # Re-rank top-k for display (consistent with policy posterior)
    idx = np.argsort(posterior_for_policy)[::-1][:k]
    topk = [(art["label_encoder"].inverse_transform([i])[0], float(posterior_for_policy[i])) for i in idx]

    # --- 3) Choose next question candidates by expected usefulness ---
    candidates = choose_next_question_info_gain(
        session=session,
        disease_proba=posterior_for_policy,
        p_e_given_d=art["p_e_given_d"],
        evidence_bases=art["evidence_bases"],
        evidences_map=art["evidences_map"],
        top_n=top_n_questions,
    )

    chosen = candidates[0] if candidates else None

    next_q = None
    if chosen is not None:
        base, score, p_yes, q_text = chosen
        meta = art["evidences_map"].get(base, {}) or {}
        next_q = {
            "base": base,
            "question_en": (meta.get("question_en") or q_text or base).strip(),
            "data_type": meta.get("data_type"),
            "default_value": meta.get("default_value"),
            "p_yes_dataset": float(p_yes),
            "usefulness_score": float(score),  # internal only; avoid ML jargon in user-facing text
        }

    findings = [token_to_readable(t, art["evidences_map"]) for t in sorted(session.pos_items)]

    return {
        "findings": findings,
        "topk": topk,
        "topk_model": topk_model,
        "posterior_model": np.asarray(posterior_model, dtype=float),
        "posterior_for_policy": posterior_for_policy,
        "candidates": candidates,      # list[(base, score, p_yes, q_text)]
        "chosen": chosen,              # tuple or None
        "next_question": next_q,        # dict or None
        "session": session,
    }


In [22]:
# ============================================================
# 4.3 Smoke-demo: run one assistant turn (quick QC)
# ============================================================

# Demo session (example positives; adjust freely)
session = DiagnosisSession(age=45.0, sex="F")
session.add_positive("E_66")
session.add_positive("E_58=3")
session.add_positive("E_58=7")

_tmp_art = {
    "model_token": model_token,
    "mlb_token": mlb_token,
    "ohe": ohe,
    "scaler": scaler,
    "label_encoder": label_encoder,
    "feature_spec": feature_spec,
    "p_e_given_d": p_e_given_d,
    "evidence_bases": evidence_bases,
    "evidence_index": evidence_index,
    "evidences_map": evidences_map,
    "conditions_map": conditions_map,
}

out = assistant_turn(session, art=_tmp_art, k=5, top_n_questions=6)

print("Top-k conditions:")
for name, p in out["topk"]:
    print(f"  {name:35s}  {p:.4f}")

print("\nNext-question candidates (top):")
if out["chosen"] is None:
    print("  (no candidates left)")
else:
    base, score, p_yes, q_text = out["chosen"]
    print(f"  base={base} | p_yes≈{p_yes:.2f} | usefulness_score≈{score:.4f}")
    print(f"  q: {q_text}")


Top-k conditions:
  Guillain-Barré syndrome              0.3884
  Myocarditis                          0.1319
  Acute dystonic reactions             0.1165
  Myasthenia gravis                    0.0882
  Bronchiectasis                       0.0844

Next-question candidates (top):
  base=E_53 | p_yes≈0.21 | usefulness_score≈0.4687
  q: Do you have pain somewhere, related to your reason for consulting?


## Section 5 — Session updates and multi‑turn loop demo

So far, retrieval and explanation assembly are in place, and one assistant “turn” can propose:
- Top‑k diagnoses (with probabilities)
- The next best question to ask
- A grounded explanation in plain language

In this section we add the missing glue for a real interaction:
1) take an answer to the chosen question,
2) update the `DiagnosisSession` (positive / negative / unknown),
3) run the next turn again.

This makes it possible to run a short multi‑turn demo trace for the Zoom presentation.

In [23]:
# ============================================================
# 5.0 Bundle runtime artifacts into a single dict (ART)
#     - Used by assistant_turn(...) and explanation functions
# ============================================================

from typing import Dict, Any

# Deterministic mapping: base evidence code -> column index in p_e_given_d
if "evidence_index" not in globals():
    evidence_index = {base: j for j, base in enumerate(evidence_bases)}

# Load mini-manual into memory (if present on disk)
mini_manual_loaded: Dict[str, str] = {}
manual_dir = config.output_dir / "rag" / "mini_manual"
index_path = manual_dir / "index.json"

if index_path.exists():
    idx = json.loads(index_path.read_text(encoding="utf-8"))
    for cond_name, fname in idx.items():
        p = manual_dir / fname
        if p.exists():
            mini_manual_loaded[cond_name] = p.read_text(encoding="utf-8")

print(f"✓ mini_manual loaded: {len(mini_manual_loaded)} condition docs")

# Phase 2 check: confirm key format matches the model's condition names
if mini_manual_loaded:
    label_names = set(getattr(label_encoder, "classes_", []))
    manual_names = set(mini_manual_loaded.keys())

    missing = sorted(label_names - manual_names)
    extra = sorted(manual_names - label_names)

    if missing or extra:
        print("⚠ mini_manual keys do not match label_encoder.classes_.")
        print(f"  missing: {len(missing)}")
        print(f"  extra:   {len(extra)}")
    else:
        print("✓ mini_manual keys match label_encoder.classes_ (e.g., 'Influenza').")

ART: Dict[str, Any] = {
    # ML artifacts
    "model_token": model_token,
    "mlb_token": mlb_token,
    "ohe": ohe,
    "scaler": scaler,
    "label_encoder": label_encoder,
    "feature_spec": feature_spec,
    # Policy artifacts
    "p_e_given_d": p_e_given_d,
    "evidence_bases": evidence_bases,
    "evidence_index": evidence_index,
    # Maps
    "evidences_map": evidences_map,
    "conditions_map": conditions_map,
    # RAG / retrieval artifacts
    "retriever": retr,
    "retrieve_snippets": retrieve_snippets,
    # Optional mini-manual
    "mini_manual": mini_manual_loaded,
}

print("✓ ART bundle ready")
print("  - token vocab:", len(ART["mlb_token"].classes_))
print("  - evidence bases:", len(ART["evidence_bases"]))
print("  - p_e_given_d shape:", ART["p_e_given_d"].shape)
print("  - retriever docs:", len(ART["retriever"]["docs"]))


✓ mini_manual loaded: 49 condition docs
✓ mini_manual keys match label_encoder.classes_ (e.g., 'Influenza').
✓ ART bundle ready
  - token vocab: 972
  - evidence bases: 223
  - p_e_given_d shape: (49, 223)
  - retriever docs: 321


In [24]:
# ============================================================
# 5.1 Session update: apply an answer for one base evidence
#      - B (binary): yes/no/unknown
#      - C (0..10 style numeric categories): store token base=value
#      - M (multi-choice V_*): store token base=V_*
#      - default_value is always used (never hard-coded)
# ============================================================

from typing import Any, Iterable, List, Optional, Tuple, Union

def _as_list(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    return [x]

def _normalize_yes_no_unknown(x: Any) -> str:
    """Return 'yes'|'no'|'unknown' from flexible inputs."""
    if x is None:
        return "unknown"
    if isinstance(x, bool):
        return "yes" if x else "no"
    s = str(x).strip().lower()
    if s in {"y", "yes", "true", "1"}:
        return "yes"
    if s in {"n", "no", "false", "0"}:
        return "no"
    if s in {"?", "unk", "unknown", "na", "n/a", ""}:
        return "unknown"
    # fallback: treat unrecognized input as unknown (safer)
    return "unknown"

def apply_answer_to_session(
    session: DiagnosisSession,
    base: str,
    answer: Any,
    evidences_map: dict,
    *,
    mlb_token_vocab: Optional[set] = None,
) -> Tuple[str, List[str]]:
    """
    Update session state from an answer.

    Returns:
      (status, tokens_added)
        status in {'positive','negative','unknown','skipped'}
    """
    meta = evidences_map.get(base)
    if meta is None:
        session.add_unknown(base)
        return "unknown", []

    dtype = meta.get("data_type")
    default_val = meta.get("default_value")

    tokens_added: List[str] = []

    # -----------------------
    # Binary evidence (B)
    # -----------------------
    if dtype == "B":
        ynu = _normalize_yes_no_unknown(answer)
        if ynu == "yes":
            session.add_positive(base)
            tokens_added.append(base)
            return "positive", tokens_added
        if ynu == "no":
            session.add_negative(base)
            return "negative", tokens_added
        session.add_unknown(base)
        return "unknown", tokens_added

    # -----------------------
    # Numeric categorical (C)
    # Example tokens: E_58=3
    # -----------------------
    if dtype == "C":
        if answer is None:
            session.add_unknown(base)
            return "unknown", tokens_added

        # Convert to a clean string token value (prefer int-like strings)
        try:
            val_num = float(answer)
            val = str(int(val_num)) if abs(val_num - int(val_num)) < 1e-9 else str(val_num)
        except Exception:
            # fallback: keep raw
            val = str(answer).strip()

        # If value equals default_value, treat as "negative-ish" (baseline) and do not add token
        if str(val) == str(default_val):
            session.add_negative(base)
            return "negative", tokens_added

        token = f"{base}={val}"
        if (mlb_token_vocab is None) or (token in mlb_token_vocab):
            session.add_positive(token)
            tokens_added.append(token)
            return "positive", tokens_added

        # Token not in vocab -> still mark asked, but do not add as positive ML input
        session.add_unknown(base)
        return "skipped", tokens_added

    # -----------------------
    # Multi-choice (M)
    # Example tokens: E_55=V_29
    # -----------------------
    if dtype == "M":
        values = _as_list(answer)
        if not values:
            session.add_unknown(base)
            return "unknown", tokens_added

        # Support comma-separated strings
        if len(values) == 1 and isinstance(values[0], str) and "," in values[0]:
            values = [v.strip() for v in values[0].split(",") if v.strip()]

        # If the user picks only default (e.g., V_123 "nowhere"), treat as negative
        # Otherwise, add one token per selected value.
        cleaned = [str(v).strip() for v in values if v is not None]
        cleaned = [v for v in cleaned if v != ""]

        if len(cleaned) == 0:
            session.add_unknown(base)
            return "unknown", tokens_added

        if len(cleaned) == 1 and str(cleaned[0]) == str(default_val):
            session.add_negative(base)
            return "negative", tokens_added

        # Add all non-default selections as positive tokens
        for v in cleaned:
            if str(v) == str(default_val):
                continue
            token = f"{base}={v}"
            if (mlb_token_vocab is None) or (token in mlb_token_vocab):
                session.add_positive(token)
                tokens_added.append(token)

        if tokens_added:
            return "positive", tokens_added

        session.add_unknown(base)
        return "skipped", tokens_added

    # Unknown dtype: mark asked but unknown
    session.add_unknown(base)
    return "unknown", tokens_added

In [25]:
# ============================================================
# 5.2 Multi-turn demo trace runner (presentation helper)
#     - Calls explanation functions each turn
#     - Saves outputs to disk for later visual inspection
# ============================================================

from pprint import pprint
from pathlib import Path


def run_demo_trace(
    *,
    age: float,
    sex: str,
    initial_positive_tokens: List[str],
    scripted_answers: List[Any],
    k: int = 5,
    top_n_questions: int = 6,
    save_path: Path | None = None,
):
    """Run a short scripted trace (deterministic; educational)."""
    session = DiagnosisSession(age=age, sex=sex)

    vocab = set(ART["mlb_token"].classes_)

    # Add initial positives (must be valid token strings for mlb_token)
    for tok in initial_positive_tokens:
        if tok not in vocab:
            # still allow adding (session stores it), but warn: it would be ignored by the model
            print(f"⚠ token not in vocab (will be ignored by ML): {tok}")
        session.add_positive(tok)

    if save_path is None:
        save_path = config.output_dir / "rag" / "demo_trace_log.json"
    save_path.parent.mkdir(parents=True, exist_ok=True)

    trace_log: List[Dict[str, Any]] = []

    print("=== DEMO TRACE START ===")
    print("Initial positives:", sorted(session.pos_items))

    for step in range(len(scripted_answers)):
        print(f"\n--- Turn {step+1} ---")
        out = assistant_turn(session, art=ART, k=k, top_n_questions=top_n_questions)

        # 1) Print Top-k
        print("Top diagnoses:")
        for name, p in out["topk"]:
            print(f"  - {name:35s}  {p:.3f}")

        # 2) Explanations (required for the final project)
        exp_topk = explain_topk_diagnoses(session, out["topk"], art=ART)
        exp_next = explain_next_question(session, out["candidates"], out["topk"], art=ART)

        print("\n[Explanation] Why these Top‑k diagnoses now?")
        print(exp_topk)

        print("\n[Explanation] Why ask this next question?")
        print(exp_next)

        # 3) Decide next base to answer
        next_q = out.get("next_question")
        if not isinstance(next_q, dict) or not next_q.get("base"):
            print("Stopping: no next question available.")
            break

        base = str(next_q["base"])
        q_en = str(next_q.get("question_en") or base)
        print(f"\nNext question: {base} | {q_en}")

        # Snapshot before applying the scripted answer (for logging)
        positives_before = sorted(session.pos_items)
        negatives_before = sorted(session.neg_bases)
        asked_before = sorted(session.asked_bases)


        # 4) Apply scripted answer to session
        ans = scripted_answers[step]
        status, tokens_added = apply_answer_to_session(
            session,
            base,
            ans,
            ART["evidences_map"],
            mlb_token_vocab=vocab,
        )
        print(f"Answer applied: {ans!r} -> status={status}, tokens_added={tokens_added}")

        positives_after = sorted(session.pos_items)
        negatives_after = sorted(session.neg_bases)
        asked_after = sorted(session.asked_bases)


        trace_log.append(
            {
                "turn": step + 1,
                "age": age,
                "sex": sex,
                "positives_before": positives_before,
                "negatives_before": negatives_before,
                "topk": out["topk"],
                "next_base": base,
                "next_question_en": q_en,
                "scripted_answer": ans,
                "answer_status": status,
                "tokens_added": tokens_added,
                "positives_after": positives_after,
                "negatives_after": negatives_after,
                "asked_bases_before": asked_before,
                "asked_bases_after": asked_after,
                "explain_topk_diagnoses": exp_topk,
                "explain_next_question": exp_next,
            }
        )

    # Persist trace to disk (Fix 12)
    with save_path.open("w", encoding="utf-8") as f:
        json.dump(trace_log, f, indent=2, ensure_ascii=False)

    print("\n=== DEMO TRACE END ===")
    print("Final positives:", sorted(session.pos_items))
    print("Asked bases:", len(session.asked_bases))
    print(f"✓ Saved demo trace log: {save_path}")

    return {"session": session, "trace_log": trace_log, "save_path": str(save_path)}


In [26]:
# ============================================================
# 5.3 Example demo run (edit freely for rehearsal)
# ============================================================

# These are example tokens similar to the smoke test style (must exist in mlb_token vocab).
initial_tokens = ["E_66", "E_58=3", "E_58=7"]

# Scripted answers correspond to the questions the policy will pick.
# For B questions: "yes"/"no"/"unknown"
# For C questions: numbers like 0..10
# For M questions: values like "V_29" or lists like ["V_29","V_122"]
scripted = ["no", "yes", "unknown"]

_ = run_demo_trace(
    age=45.0,
    sex="F",
    initial_positive_tokens=initial_tokens,
    scripted_answers=scripted,
    k=5,
    top_n_questions=6,
)

=== DEMO TRACE START ===
Initial positives: ['E_58=3', 'E_58=7', 'E_66']

--- Turn 1 ---
Top diagnoses:
  - Guillain-Barré syndrome              0.388
  - Myocarditis                          0.132
  - Acute dystonic reactions             0.117
  - Myasthenia gravis                    0.088
  - Bronchiectasis                       0.084

[Explanation] Why these Top‑k diagnoses now?
Educational explanation (not medical advice).
What we know so far (positive findings):
  - How precisely is the pain located? (value: 3)
  - How precisely is the pain located? (value: 7)
  - Are you experiencing shortness of breath or difficulty breathing in a significant way? (reported)

Why we currently rank these diagnoses near the top (dataset-grounded):
1) Guillain-Barré syndrome (estimated likelihood ≈ 0.388)
   Mini-manual: **Overview:** Guillain-Barré syndrome (GBS) is an acute autoimmune disorder in which the immune system attacks the peripheral nerves. It is usually triggered by a preceding infecti

### 5.4 Demo: finding an ambiguous starting state


In [27]:
# ============================================================
# 5.4.3 Demo: finding an ambiguous starting state (oracle simulation)
# ============================================================

from typing import Any, Dict, List, Optional, Set
import json
import numpy as np

from src.policy import entropy
from src.types import parse_token


def condition_base_set(condition_name: str, conditions_map: Dict[str, Any]) -> Set[str]:
    meta = conditions_map[condition_name]
    s = set((meta.get("symptoms") or {}).keys())
    a = set((meta.get("antecedents") or {}).keys())
    return s | a


def find_ambiguous_seed_for_condition(
    condition_name: str,
    *,
    n_trials: int = 800,
    m_min: int = 2,
    m_max: int = 3,
    top1_max: float = 0.85,
    seed: int = 0,
) -> Optional[Dict[str, Any]]:
    """Search for a small set of positive tokens that keeps the posterior relatively uncertain."""
    rng = np.random.default_rng(seed)

    if condition_name not in set(ART["label_encoder"].classes_):
        raise ValueError(f"Condition '{condition_name}' not found in label_encoder.classes_.")

    target_idx = int(ART["label_encoder"].transform([condition_name])[0])
    bases = sorted(list(condition_base_set(condition_name, ART["conditions_map"])))

    token_vocab = set(ART["mlb_token"].classes_)

    best: Optional[Dict[str, Any]] = None
    best_score = -1.0

    for _ in range(n_trials):
        m = int(rng.integers(m_min, m_max + 1))
        chosen_bases = rng.choice(bases, size=m, replace=False)

        tokens: List[str] = []
        ok = True
        for b in chosen_bases:
            tok = valid_positive_token_for_base(b, ART["evidences_map"], token_vocab)
            if tok is None:
                ok = False
                break
            tokens.append(tok)
        if not ok:
            continue

        sess = DiagnosisSession(age=45.0, sex="F")
        for tok in tokens:
            sess.add_positive(tok)

        _, proba = predict_topk_from_session_token(
            sess,
            ART["model_token"],
            mlb_token=ART["mlb_token"],
            ohe=ART["ohe"],
            scaler=ART["scaler"],
            label_encoder=ART["label_encoder"],
            feature_spec=ART["feature_spec"],
            k=5,
        )

        proba = proba / (proba.sum() + 1e-12)
        H = entropy(proba)
        top1 = float(np.max(proba))
        p_target = float(proba[target_idx])

        if top1 > top1_max:
            continue

        # Score: prefer uncertainty + target still somewhat plausible
        score = H + 0.5 * p_target
        if score > best_score:
            best_score = score
            best = {
                "tokens": tokens,
                "entropy": float(H),
                "top1": top1,
                "p_target": p_target,
            }

    return best


def simulate_answer_from_condition(base: str, condition_name: str) -> str:
    """Simple oracle: YES if base is listed for the condition in release_conditions.json; else NO."""
    base_set = condition_base_set(condition_name, ART["conditions_map"])
    return "yes" if base in base_set else "no"


def run_ambiguous_seed_demo(
    condition_name: str,
    *,
    max_turns: int = 5,
    seed: int = 0,
) -> Dict[str, Any]:
    """Run a short oracle-driven demo starting from an intentionally ambiguous seed.

    Phase 2 requirement:
    - We run the full max_turns (no early-stop) to guarantee ≥5 turns for validation/logging.
    """
    seed_info = find_ambiguous_seed_for_condition(condition_name, seed=seed)

    # Robust fallback: if we fail to find an "ambiguous" seed, we still run a deterministic demo
    # so the notebook remains reproducible and the explanation outputs can be inspected.
    if seed_info is None:
        print("⚠ No ambiguous seed found with current settings. Falling back to a simple deterministic seed.")
        bases = sorted(list(condition_base_set(condition_name, ART["conditions_map"])))
        token_vocab = set(ART["mlb_token"].classes_)

        tokens: List[str] = []
        for b in bases:
            tok = valid_positive_token_for_base(b, ART["evidences_map"], token_vocab)
            if tok is not None:
                tokens.append(tok)
            if len(tokens) >= 2:
                break

        if not tokens:
            print("No valid token seed could be constructed; cannot run the oracle demo.")
            return {"ok": False, "reason": "no_valid_seed_tokens"}

        seed_info = {"tokens": tokens, "entropy": None, "top1": None, "p_target": None}

    session = DiagnosisSession(age=45.0, sex="F")
    for tok in seed_info["tokens"]:
        session.add_positive(tok)

    vocab = set(ART["mlb_token"].classes_)

    print("=== DEMO TRACE START ===")
    print("Target condition (oracle):", condition_name)
    print("Initial positives:", sorted(session.pos_items))

    if seed_info.get("entropy") is not None:
        print(
            f"Seed diagnostics: entropy={seed_info['entropy']:.3f}, "
            f"top1={seed_info['top1']:.3f}, p_target={seed_info['p_target']:.3f}"
        )
    else:
        print("Seed diagnostics: (fallback seed; ambiguity metrics not computed)")
    print()

    log: List[Dict[str, Any]] = []

    for t in range(1, max_turns + 1):
        out = assistant_turn(session, art=ART, k=5, top_n_questions=6)

        print(f"--- Turn {t} ---")
        print("Known findings:")
        for f in out["findings"][:10]:
            print("  -", f)

        print("\nTop diagnoses:")
        for name, p in out["topk"]:
            print(f"  - {name:35s} {p:.3f}")

        # Explanations (same functions as the main demo)
        exp_topk = explain_topk_diagnoses(session, out["topk"], art=ART)
        exp_next = explain_next_question(session, out["candidates"], out["topk"], art=ART)

        print("\n[Explanation] Why these Top‑k diagnoses now?")
        print(exp_topk)

        print("\n[Explanation] Why ask this next question?")
        print(exp_next)

        nq = out["next_question"]
        if nq is None:
            print("\nStopping: no next question available.")
            break

        base = str(nq["base"])
        qtext = str(nq["question_en"])
        ans = simulate_answer_from_condition(base, condition_name)

        print("\nNext question:")
        print(" ", qtext)
        print("Simulated answer:", ans)

        if ans == "yes":
            tok = valid_positive_token_for_base(base, ART["evidences_map"], vocab)
            if tok is None:
                session.add_unknown(base)
                status = "unknown"
                tokens_added: List[str] = []
                print("  (No valid token found for this base in vocab; recorded as unknown.)")
            else:
                _, val = parse_token(tok)
                answer_value = ("yes" if val is None else val)
                status, tokens_added = apply_answer_to_session(
                    session,
                    base,
                    answer_value,
                    ART["evidences_map"],
                    mlb_token_vocab=vocab,
                )
        else:
            status, tokens_added = apply_answer_to_session(
                session,
                base,
                "no",
                ART["evidences_map"],
                mlb_token_vocab=vocab,
            )

        log.append(
            {
                "turn": t,
                "seed_tokens": seed_info["tokens"],
                "topk": out["topk"],
                "next_base": base,
                "next_question_en": qtext,
                "oracle_answer": ans,
                "answer_status": status,
                "tokens_added": tokens_added,
                "explain_topk_diagnoses": exp_topk,
                "explain_next_question": exp_next,
            }
        )

        print()

    # Save for inspection (same folder as the main demo trace)
    out_path = config.output_dir / "rag" / "ambiguous_seed_demo_log.json"
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(log, f, indent=2, ensure_ascii=False)

    print("=== DEMO TRACE END ===")
    print("Final positives:", sorted(session.pos_items))
    print("Asked bases:", len(session.asked_bases))
    print(f"✓ Saved ambiguous demo log: {out_path}")

    return {"ok": True, "seed_info": seed_info, "log": log, "save_path": str(out_path)}


# Pick a target that often has overlap with others (more interesting than very specific tokens)
ambiguous_demo = run_ambiguous_seed_demo("Influenza", max_turns=5, seed=2)


=== DEMO TRACE START ===
Target condition (oracle): Influenza
Initial positives: [np.str_('E_227'), np.str_('E_53'), np.str_('E_79')]
Seed diagnostics: entropy=2.741, top1=0.155, p_target=0.000

--- Turn 1 ---
Known findings:
  - Are you immunosuppressed? (reported)
  - Do you have pain somewhere, related to your reason for consulting? (reported)
  - Do you smoke cigarettes? (reported)

Top diagnoses:
  - Guillain-Barré syndrome             0.155
  - Pulmonary neoplasm                  0.144
  - Bronchiectasis                      0.128
  - Pancreatic neoplasm                 0.087
  - Acute dystonic reactions            0.074

[Explanation] Why these Top‑k diagnoses now?
Educational explanation (not medical advice).
What we know so far (positive findings):
  - Are you immunosuppressed? (reported)
  - Do you have pain somewhere, related to your reason for consulting? (reported)
  - Do you smoke cigarettes? (reported)

Why we currently rank these diagnoses near the top (dataset-grounded

In [28]:
# ============================================================
# 5.4.4 Visual validation: inspect explanation outputs (Influenza demo)
# ============================================================

import json
from pathlib import Path
import pandas as pd


def _head(text: str, n_lines: int = 12) -> str:
    lines = (text or "").splitlines()
    return "\n".join(lines[:n_lines]).strip()


log_path = Path(config.output_dir) / "rag" / "ambiguous_seed_demo_log.json"
assert log_path.exists(), (
    f"Expected demo log at {log_path}. "
    "Run Section 5.4.3 first to generate the 5-turn Influenza trace."
)

log = json.loads(log_path.read_text(encoding="utf-8"))

# Compact table for quick scanning (flag empties / missing enrichments)
rows = []
for r in log:
    top1_name, top1_p = (None, None)
    if r.get("topk"):
        top1_name = r["topk"][0][0]
        top1_p = float(r["topk"][0][1])

    exp_top = str(r.get("explain_topk_diagnoses", "") or "")
    exp_next = str(r.get("explain_next_question", "") or "")

    rows.append(
        {
            "turn": int(r.get("turn", 0)),
            "top1": top1_name,
            "top1_p": top1_p,
            "next_base": r.get("next_base"),
            "oracle_answer": r.get("oracle_answer"),
            "exp_topk_chars": len(exp_top),
            "exp_next_chars": len(exp_next),
            "has_mini_manual": ("Mini-manual:" in exp_top),
            "has_retrieved_snippet": ("Retrieved context" in exp_top) or ("Evidence definition" in exp_next),
        }
    )

df = pd.DataFrame(rows).sort_values("turn")
display(df)

# Flag any empty outputs (should not happen)
empty_topk = df[df["exp_topk_chars"] == 0]["turn"].tolist()
empty_next = df[df["exp_next_chars"] == 0]["turn"].tolist()

if empty_topk:
    print(f"⚠ Empty explain_topk_diagnoses output on turns: {empty_topk}")
if empty_next:
    print(f"⚠ Empty explain_next_question output on turns: {empty_next}")

# Preview the start of each explanation block (human validation)
for r in log:
    t = r.get("turn")
    print(f"\n=== Turn {t} — explanation preview ===")
    print("[Top‑k explanation]")
    print(_head(str(r.get("explain_topk_diagnoses", "")), n_lines=14) or "(empty)")
    print("\n[Next-question explanation]")
    print(_head(str(r.get("explain_next_question", "")), n_lines=14) or "(empty)")


,turn,top1,top1_p,next_base,oracle_answer,exp_topk_chars,exp_next_chars,has_mini_manual,has_retrieved_snippet
0,1,Guillain-Barré syndrome,0.1552,E_54,yes,5332,1036,True,True
1,2,Spontaneous pneumothorax,0.2311,E_55,yes,5511,1057,True,True
2,3,Spontaneous pneumothorax,0.2775,E_56,yes,5584,833,True,True
3,4,Spontaneous pneumothorax,0.2962,E_220,no,5642,910,True,True
4,5,Pancreatic neoplasm,0.2133,E_58,yes,5642,843,True,True



=== Turn 1 — explanation preview ===
[Top‑k explanation]
Educational explanation (not medical advice).
What we know so far (positive findings):
  - Are you immunosuppressed? (reported)
  - Do you have pain somewhere, related to your reason for consulting? (reported)
  - Do you smoke cigarettes? (reported)

Why we currently rank these diagnoses near the top (dataset-grounded):
1) Guillain-Barré syndrome (estimated likelihood ≈ 0.155)
   Mini-manual: **Overview:** Guillain-Barré syndrome (GBS) is an acute autoimmune disorder in which the immune system attacks the peripheral nerves. It is usually triggered by a preceding infection (Campylobacter jejuni, EBV, influenza). It affects the peripheral nervous system, causing rapidly progressive weakness. **Typical Presentation:** Ascending, symmetric muscle weakness starting in the legs and progressing upward over days. Deep tendon reflexes are diminished or absent. Paresthesias, back pain, and autonomic dysfunction (blood pressure instability

# Notebook 03 — Summary

In this notebook we added a **local, fully offline retrieval + explainability layer** on top of the already‑trained token‑level predictor. The assistant loop is now able to (1) rank a differential, (2) choose the next question to ask, and (3) explain both decisions using **dataset-grounded** context from the DDXPlus mapping files and the saved mini‑manual documents (optionally enriched with cited clinical summaries).

## Success criteria status

- ✅ **RAG pipeline runs end‑to‑end**: a `DiagnosisSession` can be created, passed through `assistant_turn(...)`, and the assistant returns a ranked differential plus a next question and explanations.
- ✅ **Questions are phrased in natural language and grounded in the dataset**: the assistant surfaces `question_en` (and related metadata) directly from `release_evidences.json`, with no hard‑coded wording.
- ✅ **Top‑k differential is explained with supporting context**: for each ranked condition, we surface a short **mini‑manual blurb** (or a TF‑IDF retrieved snippet as fallback) plus explicit overlap with the DDXPlus symptom/antecedent mappings.
- ✅ **Session loop completes ≥5 turns without errors**: the Influenza oracle demo in Section 5.4.3 runs for the full `max_turns=5` and saves a turn‑by‑turn trace for inspection.
- ✅ **Outputs are logged and saved to disk**: the demo traces save the full Top‑k lists, chosen questions, scripted/oracle answers, and both explanation blocks into JSON logs under `outputs/rag/`.

## Artifacts saved

The notebook writes the following artifacts using stable, submission‑friendly paths:

- **Mini‑manual (Markdown)**  
  - `outputs/rag/mini_manual/index.json`  
  - `outputs/rag/mini_manual/*.md` (49 files; one per condition)
- **Canonical TF‑IDF retriever**  
  - `outputs/rag/tfidf_retriever.joblib` (321 docs: 223 evidences + 49 conditions + 49 manuals)
- **Demo / validation logs (include full explanations)**  
  - `outputs/rag/demo_trace_log.json`  
  - `outputs/rag/ambiguous_seed_demo_log.json`

## What we intentionally did not do in this notebook

To keep the project reproducible, offline, and aligned with the architecture decisions:

- We did **not** add dense-embedding retrieval or any external vector database.
- We did **not** make any live calls to PubMed / web services or any LLM API.
- We did **not** retrain models; all inference uses the already‑saved token predictor artifacts.

## Next steps

1. **Refactor notebook helpers into `src/rag.py`** (retrieval wrapper, mini‑manual utilities, explanation formatting, demo runners).
2. **Build a Streamlit demo app**: symptom input → iterative Q&A → ranked diagnoses with explanations → safety disclaimer.
3. **Prepare the final presentation**: include a short recorded trace (log‑backed), artifact paths, and a reproducibility checklist.

**Safety disclaimer:** This notebook is for education and model-behavior explanation only. It is not medical advice and must not be used for diagnosis or treatment decisions.
